In [ ]:
import hashlib, json, logging, os, platform, sys, time, urllib.request, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm, catboost, joblib
from joblib import Parallel, delayed
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

# ---- paths ------------------------------------------------------------------------------
ROOT      = Path.cwd()
RAW_DIR   = ROOT / "data" / "raw"
EXT_DIR   = ROOT / "data" / "external"
OUT_DIR   = ROOT / "outputs"
REF_DIR   = ROOT / "reference"
OUT_DIR.mkdir(parents=True, exist_ok=True)

APHRC_PATH = EXT_DIR / "IMHDSS_Climate_Mortality.csv"
APHRC_URL  = ("https://huggingface.co/datasets/APHRC/Climate_and_Health/"
              "resolve/main/IMHDSS_Climate_Mortality.csv")

# Checksums of the exact inputs/outputs used for the submitted file. A mismatch on the
# Zindi files only WARNS (the host may re-export them); the APHRC file is verified strictly.
MD5 = {
    "Train.csv":                     "8a981e0902de98af0e46cebcfc9f25a2",
    "Test.csv":                      "9d5ce2f74ecd1ce65cc38b500df08821",
    "SampleSubmission.csv":          "29fadb8ab7c8750e295a3a2373a31051",
    "IMHDSS_Climate_Mortality.csv":  "29e902aa6d1fcf659be9d6c4a1a20683",
    "submission_v8.csv":             "ec6a98f9ace4f8e9a3c982fd8d0e34c7",   # stage-1 output
    "submission_v10.csv":            "e5d3086d37ab847b5f975054cbb94286",   # final output
}

# ---- modelling configuration -------------------------------------------------------------
N_SPLITS    = 5                      # StratifiedGroupKFold folds (groups = village)
FOLD_SEEDS  = list(range(500, 520))  # 20 repeated-CV fold seeds
MODEL_SEEDS = (42, 7, 2026)          # 3 model seeds per fold -> seed = s*100 + fold_seed
THRESHOLD   = 0.5                    # TargetF1 = proba >= 0.5 (F1 optimum, see §8)
N_JOBS      = int(os.environ.get("N_JOBS", min(os.cpu_count() or 1, len(FOLD_SEEDS))))

# ---- logging -----------------------------------------------------------------------------
log = logging.getLogger("v10")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
for _h in (logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "run.log", mode="w")):
    _h.setFormatter(_fmt); log.addHandler(_h)

# ---- stage timer -------------------------------------------------------------------------
RUNTIME = {}
@contextmanager
def stage(name):
    t0 = time.perf_counter()
    log.info(f"▶ {name}")
    try:
        yield
    except Exception:
        log.exception(f"✖ {name} failed")
        raise
    RUNTIME[name] = time.perf_counter() - t0
    log.info(f"✔ {name}  ({RUNTIME[name]:.1f}s)")

def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

T_START = time.perf_counter()
log.info(f"python {platform.python_version()} | {platform.platform()} | cpus={os.cpu_count()} | N_JOBS={N_JOBS}")
log.info(" | ".join(f"{m.__name__} {m.__version__}" for m in
                    (np, pd, sklearn, xgboost, lightgbm, catboost, joblib)))

15:30:30 | INFO    | python 3.12.13 | macOS-27.0.1-arm64-arm-64bit | cpus=10 | N_JOBS=10


15:30:30 | INFO    | numpy 2.5.2 | pandas 3.0.5 | sklearn 1.9.0 | xgboost 3.4.1 | lightgbm 4.7.0 | catboost 1.2.10 | joblib 1.5.3


In [ ]:
REQ_COMP  = {"ID","zone","gender","deathdate","age","avg_temperature","max_temperature",
             "min_temperature","precipitation","latitude","longitude","location"}
REQ_APHRC = {"gender","deathdate","age","avg_temperature","max_temperature",
             "min_temperature","precipitation","causeofdeath","GeneralCoD"}

def check_md5(path, strict=False):
    got, exp = md5(path), MD5.get(Path(path).name)
    if exp is None or got == exp:
        log.info(f"  {Path(path).name:30s} md5 OK ({got})")
        return
    msg = f"{Path(path).name}: md5 {got} != expected {exp}"
    if strict:
        raise ValueError(msg + " -- file differs from the one used for the submission")
    log.warning(msg + " -- results may differ from the submitted file")

def fetch_aphrc(path=APHRC_PATH, url=APHRC_URL):
    # Download the APHRC verbal-autopsy sample if it is not already present.
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    log.info(f"  downloading {url}")
    try:
        tmp = path.with_suffix(".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    except Exception as e:
        raise RuntimeError(
            f"Could not download the APHRC file ({e}). Download it manually from\n  {url}\n"
            f"and save it as {path}") from e
    return path

def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {path}. Download the competition data from Zindi and place "
            f"Train.csv, Test.csv and SampleSubmission.csv in {RAW_DIR}/")
    return path

with stage("extract"):
    for f in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        check_md5(require_file(RAW_DIR / f))
    check_md5(fetch_aphrc(), strict=True)

    tr = pd.read_csv(RAW_DIR / "Train.csv", parse_dates=["deathdate"])
    te = pd.read_csv(RAW_DIR / "Test.csv",  parse_dates=["deathdate"])
    ss = pd.read_csv(RAW_DIR / "SampleSubmission.csv")
    ap = pd.read_csv(APHRC_PATH)

    for name, df, req in (("Train", tr, REQ_COMP | {"is_climate_sensitive"}),
                          ("Test", te, REQ_COMP), ("APHRC", ap, REQ_APHRC)):
        missing = req - set(df.columns)
        assert not missing, f"{name} is missing required columns: {sorted(missing)}"
    assert tr.ID.is_unique and te.ID.is_unique, "duplicate IDs in competition data"
    assert set(ss.ID) == set(te.ID) and len(ss) == len(te), "SampleSubmission IDs != Test IDs"
    assert tr.is_climate_sensitive.isin([0, 1]).all(), "target must be binary"
    assert not tr[list(REQ_COMP)].isna().any().any(), "unexpected NaNs in Train inputs"
    assert not te[list(REQ_COMP)].isna().any().any(), "unexpected NaNs in Test inputs"

    log.info(f"  train {tr.shape} | test {te.shape} | aphrc {ap.shape} | "
             f"target base rate {tr.is_climate_sensitive.mean():.3f}")
tr.head(3)

15:30:30 | INFO    | ▶ extract


15:30:30 | INFO    |   Train.csv                      md5 OK (8a981e0902de98af0e46cebcfc9f25a2)


15:30:30 | INFO    |   Test.csv                       md5 OK (9d5ce2f74ecd1ce65cc38b500df08821)


15:30:30 | INFO    |   SampleSubmission.csv           md5 OK (29fadb8ab7c8750e295a3a2373a31051)


15:30:30 | INFO    |   IMHDSS_Climate_Mortality.csv   md5 OK (29e902aa6d1fcf659be9d6c4a1a20683)


15:30:30 | INFO    |   train (3146, 13) | test (1030, 12) | aphrc (505, 17) | target base rate 0.651


15:30:30 | INFO    | ✔ extract  (0.0s)


,ID,zone,gender,deathdate,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,location,is_climate_sensitive
0,ID_1FE951BD,Rural,Male,2007-10-02,30.0,21.137037,25.569017,18.451642,0.951601,0.584646,33.546344,"Izimba, Iganga, Uganda",1
1,ID_5927A443,Rural,Male,2008-01-05,2.0,24.062790,29.312026,18.977212,0.053352,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
2,ID_C4E67025,Rural,Female,2008-02-14,31.0,22.407868,26.746058,18.112977,0.090349,0.573420,33.483655,"Buluza, Iganga, Uganda",1


In [ ]:
CORE = ["age", "age_log1p", "is_infant", "is_under5", "is_child", "is_elderly",
        "gender_bin", "zone_bin", "year"]
V3_COLS = ["is_age0", "is_age1_4", "post2018", "age0_x_post", "age0_x_year",
           "age14_x_post", "age5p_x_post", "age5p_x_year",
           "a5p_month_sin", "a5p_month_cos", "age_x_a5p"]
REGIME = ["sy_n", "sy_infshare", "sy_u5share", "sy_femshare", "sy_meanage",
          "y_n", "y_infshare", "y_meanage", "age_pctile_in_year", "age_minus_year_mean"]


def core_features(df):
    out = pd.DataFrame(index=df.index)
    out["age"] = df["age"]
    out["age_log1p"] = np.log1p(df["age"])
    out["is_infant"] = (df["age"] <= 1).astype(int)
    out["is_under5"] = (df["age"] < 5).astype(int)
    out["is_child"] = ((df["age"] >= 5) & (df["age"] < 18)).astype(int)
    out["is_elderly"] = (df["age"] >= 60).astype(int)
    out["gender_bin"] = (df["gender"] == "Female").astype(int)
    out["zone_bin"] = (df["zone"] == "Rural").astype(int)
    out["year"] = df.deathdate.dt.year
    return out[CORE]


def v3_features(df):
    age = df["age"].values
    yr = df["deathdate"].dt.year.values
    month = df["deathdate"].dt.month.values
    a0 = (age == 0).astype(float)
    a14 = ((age >= 1) & (age < 5)).astype(float)
    a5p = (age >= 5).astype(float)
    post = (yr >= 2018).astype(float)

    out = pd.DataFrame(index=df.index)
    out["is_age0"] = a0
    out["is_age1_4"] = a14
    out["post2018"] = post
    out["age0_x_post"] = a0 * post                     # infants: collapse after 2018
    out["age0_x_year"] = a0 * (yr - 2007)
    out["age14_x_post"] = a14 * post
    out["age5p_x_post"] = a5p * post
    out["age5p_x_year"] = a5p * (yr - 2007)
    out["a5p_month_sin"] = a5p * np.sin(2 * np.pi * month / 12)   # season only for 5+
    out["a5p_month_cos"] = a5p * np.cos(2 * np.pi * month / 12)
    out["age_x_a5p"] = a5p * age
    return out[V3_COLS]


def regime_features(tr_raw, te_raw):
    # Cohort composition per site x year and per year, on train+test INPUTS (no target).
    comb = pd.concat([tr_raw.assign(_src=0), te_raw.assign(_src=1)], ignore_index=True)
    comb["_site"] = comb.latitude.round(6).astype(str) + "_" + comb.longitude.round(6).astype(str)
    comb["_yr"] = comb.deathdate.dt.year
    comb["_u1"] = (comb.age == 0).astype(float)
    comb["_u5"] = (comb.age < 5).astype(float)
    comb["_fem"] = (comb.gender == "Female").astype(float)

    f = pd.DataFrame(index=comb.index)
    gsy = comb.groupby(["_site", "_yr"])
    for col, src in [("sy_n", "_u1"), ("sy_infshare", "_u1"), ("sy_u5share", "_u5"),
                     ("sy_femshare", "_fem"), ("sy_meanage", "age")]:
        f[col] = gsy[src].transform("size" if col == "sy_n" else "mean").values
    gy = comb.groupby("_yr")
    f["y_n"] = gy["_u1"].transform("size").values
    f["y_infshare"] = gy["_u1"].transform("mean").values
    f["y_meanage"] = gy["age"].transform("mean").values
    f["age_pctile_in_year"] = gy["age"].rank(pct=True).values
    f["age_minus_year_mean"] = comb.age.values - f["y_meanage"].values

    n = len(tr_raw)
    return f.iloc[:n].reset_index(drop=True)[REGIME], f.iloc[n:].reset_index(drop=True)[REGIME]


with stage("transform"):
    tr = tr.reset_index(drop=True); te = te.reset_index(drop=True)
    Rtr, Rte = regime_features(tr, te)
    Xtr = pd.concat([core_features(tr), v3_features(tr), Rtr], axis=1)
    Xte = pd.concat([core_features(te), v3_features(te), Rte], axis=1)
    y = tr.is_climate_sensitive.reset_index(drop=True)
    # CV groups = village ("<village>, <district>") -- test villages are ~disjoint from train
    groups = pd.factorize(tr.location.str.split(",").str[:2].str.join(","))[0]

    assert len(Xtr) == len(tr) and len(Xte) == len(te)
    assert np.isfinite(Xtr.values).all() and np.isfinite(Xte.values).all(), "non-finite features"
    log.info(f"  features: {Xtr.shape[1]} cols | villages (CV groups): {groups.max() + 1}")
Xtr.describe().T.round(3)

15:30:30 | INFO    | ▶ transform


15:30:30 | INFO    |   features: 30 cols | villages (CV groups): 38


15:30:30 | INFO    | ✔ transform  (0.0s)


,count,mean,std,min,25%,50%,75%,max
age,3146.0,24.243,31.429,0.000,0.000,3.000,49.000,110.000
age_log1p,3146.0,1.915,1.819,0.000,0.000,1.386,3.912,4.710
is_infant,3146.0,0.443,0.497,0.000,0.000,0.000,1.000,1.000
is_under5,3146.0,0.551,0.497,0.000,0.000,1.000,1.000,1.000
is_child,3146.0,0.044,0.206,0.000,0.000,0.000,0.000,1.000
is_elderly,3146.0,0.205,0.404,0.000,0.000,0.000,0.000,1.000
gender_bin,3146.0,0.477,0.500,0.000,0.000,0.000,1.000,1.000
zone_bin,3146.0,0.627,0.484,0.000,0.000,1.000,1.000,1.000
year,3146.0,2013.834,4.196,2007.000,2010.000,2013.000,2017.000,2022.000
is_age0,3146.0,0.353,0.478,0.000,0.000,0.000,1.000,1.000


In [ ]:
with stage("load"):
    Xtr.assign(ID=tr.ID, is_climate_sensitive=y).to_csv(OUT_DIR / "features_train.csv", index=False)
    Xte.assign(ID=te.ID).to_csv(OUT_DIR / "features_test.csv", index=False)
    log.info(f"  wrote outputs/features_train.csv {Xtr.shape} and outputs/features_test.csv {Xte.shape}")

15:30:30 | INFO    | ▶ load


15:30:30 | INFO    |   wrote outputs/features_train.csv (3146, 30) and outputs/features_test.csv (1030, 30)


15:30:30 | INFO    | ✔ load  (0.0s)


In [61]:
XGB_PARAMS = dict(learning_rate=.05, max_depth=3, min_child_weight=1, subsample=.85,
                  colsample_bytree=.9, reg_lambda=1.)
CAT_PARAMS = dict(learning_rate=.05, depth=4, l2_leaf_reg=3.)
LGB_PARAMS = dict(learning_rate=0.011374471419103494, num_leaves=53, min_child_samples=41,
                  subsample=0.868088589184808, subsample_freq=1,
                  colsample_bytree=0.8288246650597182,
                  reg_lambda=0.001222299102744292, reg_alpha=2.8346683989414245)

#          name        family  params      scale_pos_weight     rounds  features
MEMBERS = [("lgbm_reg", "lgbm", LGB_PARAMS, 1.3599198869839608, 263, CORE + REGIME),
           ("cat_core", "cat",  CAT_PARAMS, 1.86,               250, CORE),
           ("xgb_v3",   "xgb",  XGB_PARAMS, 1.70,               100, CORE + V3_COLS)]


def combo(y, p):
    return 0.6 * f1_score(y, (np.asarray(p) >= THRESHOLD).astype(int)) + 0.4 * roc_auc_score(y, p)


def parts(y, p):
    return f1_score(y, (np.asarray(p) >= THRESHOLD).astype(int)), roc_auc_score(y, p)


def build(fam, params, spw, rounds, seed):
    # Model with a FIXED round count, no early stopping, single-threaded (deterministic).
    if fam == "xgb":
        return xgboost.XGBClassifier(n_estimators=rounds, scale_pos_weight=spw, random_state=seed,
                                     eval_metric="logloss", verbosity=0, n_jobs=1, **params)
    if fam == "lgbm":
        return lightgbm.LGBMClassifier(n_estimators=rounds, scale_pos_weight=spw, random_state=seed,
                                       verbose=-1, n_jobs=1, **params)
    if fam == "cat":
        return catboost.CatBoostClassifier(iterations=rounds, scale_pos_weight=spw, random_seed=seed,
                                           verbose=False, allow_writing_files=False,
                                           thread_count=1, **params)
    raise ValueError(f"unknown model family {fam!r}")


def one_fold_seed(fam, params, spw, rounds, fold_seed, X, y, groups, Xte):
    # One repetition: 5 grouped folds x MODEL_SEEDS -> (OOF vector, averaged test prediction).
    import warnings; warnings.filterwarnings("ignore")
    oof, te_pred = np.zeros(len(y)), np.zeros(len(Xte))
    splits = list(StratifiedGroupKFold(N_SPLITS, shuffle=True,
                                       random_state=fold_seed).split(X, y, groups))
    for tr_i, va_i in splits:
        for s in MODEL_SEEDS:
            m = build(fam, params, spw, rounds, s * 100 + fold_seed)
            m.fit(X.iloc[tr_i], y.iloc[tr_i])
            oof[va_i] += m.predict_proba(X.iloc[va_i])[:, 1] / len(MODEL_SEEDS)
            te_pred += m.predict_proba(Xte)[:, 1] / (len(MODEL_SEEDS) * len(splits))
    return oof, te_pred


def evaluate(fam, params, spw, rounds, X, y, groups, Xte):
    # Repeated grouped CV over FOLD_SEEDS. Returns (mean, se, oofs[n_seeds, n], test_pred).
    X = pd.DataFrame(X.values, columns=list(X.columns))          # dense float64
    Xte = pd.DataFrame(Xte.values, columns=list(Xte.columns))
    res = Parallel(n_jobs=N_JOBS)(
        delayed(one_fold_seed)(fam, params, spw, rounds, fs, X, y, groups, Xte)
        for fs in FOLD_SEEDS)
    oofs = np.vstack([r[0] for r in res])
    scores = np.array([combo(y, o) for o in oofs])
    return (scores.mean(), scores.std(ddof=1) / np.sqrt(len(scores)), oofs,
            np.mean([r[1] for r in res], axis=0))


oofs, tests, rows = {}, {}, []
for name, fam, params, spw, rounds, cols in MEMBERS:
    with stage(f"train {name}"):
        mean, se, o, t = evaluate(fam, params, spw, rounds, Xtr[cols], y, groups, Xte[cols])
        oofs[name], tests[name] = o, t
        f1, auc = parts(y, o.mean(0))
        rows.append(dict(member=name, n_features=len(cols), combo=mean, se=se, F1=f1, AUC=auc))
        log.info(f"  {name:9s} combo={mean:.4f} ± {se:.4f}  F1={f1:.4f}  AUC={auc:.4f}")

06:18:15 | INFO    | ▶ train lgbm_reg


INFO:v10:▶ train lgbm_reg


06:19:48 | INFO    |   lgbm_reg  combo=0.8246 ± 0.0003  F1=0.8216  AUC=0.8306


INFO:v10:  lgbm_reg  combo=0.8246 ± 0.0003  F1=0.8216  AUC=0.8306


06:19:48 | INFO    | ✔ train lgbm_reg  (92.5s)


INFO:v10:✔ train lgbm_reg  (92.5s)


... (8 outputs left out)

06:21:26 | INFO    |   xgb_v3    combo=0.8214 ± 0.0003  F1=0.8170  AUC=0.8280


INFO:v10:  xgb_v3    combo=0.8214 ± 0.0003  F1=0.8170  AUC=0.8280


06:21:26 | INFO    | ✔ train xgb_v3  (18.2s)


INFO:v10:✔ train xgb_v3  (18.2s)


In [62]:
with stage("validate blend"):
    W = 1 / 3
    blend_oof = sum(W * oofs[n] for n in oofs)                    # [20 seeds, n_train]
    per_seed = np.array([combo(y, blend_oof[s]) for s in range(blend_oof.shape[0])])
    f1, auc = parts(y, blend_oof.mean(0))
    rows.append(dict(member="BLEND (1/3 each)", n_features=np.nan, combo=per_seed.mean(),
                     se=per_seed.std(ddof=1) / np.sqrt(len(per_seed)), F1=f1, AUC=auc))
    cv_table = pd.DataFrame(rows).set_index("member")
    cv_table.to_csv(OUT_DIR / "cv_metrics.csv")

    # OOF member correlation -- diversity is what makes an equal-weight blend pay
    corr = pd.DataFrame({n: oofs[n].mean(0) for n in oofs}).corr().round(4)
cv_table.round(4)

06:21:26 | INFO    | ▶ validate blend


INFO:v10:▶ validate blend


06:21:27 | INFO    | ✔ validate blend  (0.2s)


INFO:v10:✔ validate blend  (0.2s)


,n_features,combo,se,F1,AUC
member,,,,,
lgbm_reg,19.0,0.8246,0.0003,0.8216,0.8306
cat_core,9.0,0.8219,0.0003,0.8203,0.8279
xgb_v3,20.0,0.8214,0.0003,0.8170,0.8280
BLEND (1/3 each),NaN,0.8240,0.0003,0.8196,0.8318


In [63]:
corr

,lgbm_reg,cat_core,xgb_v3
lgbm_reg,1.0000,0.9621,0.9667
cat_core,0.9621,1.0000,0.9803
xgb_v3,0.9667,0.9803,1.0000


In [64]:
def check_submission(sub, ids):
    assert list(sub.columns) == ["ID", "TargetF1", "TargetRAUC"], "wrong columns"
    assert sub.ID.equals(ids) and len(sub) == len(ids), "IDs / order differ from Test.csv"
    assert sub.TargetF1.isin([0, 1]).all(), "TargetF1 must be 0/1"
    assert not sub.isna().any().any(), "NaNs in submission"
    assert sub.TargetRAUC.between(0, 1).all(), "TargetRAUC outside [0,1]"
    assert ((sub.TargetRAUC >= THRESHOLD).astype(int) == sub.TargetF1).all(), "F1/RAUC inconsistent"

with stage("inference stage 1"):
    proba = sum(W * tests[n] for n in tests)
    sub_v8 = pd.DataFrame({"ID": te.ID, "TargetF1": (proba >= THRESHOLD).astype(int),
                           "TargetRAUC": proba.astype(float)})
    check_submission(sub_v8, te.ID)
    V8_PATH = OUT_DIR / "submission_v8.csv"
    sub_v8.to_csv(V8_PATH, index=False)
    log.info(f"  {V8_PATH.name}: pos-rate {sub_v8.TargetF1.mean():.4f}  "
             f"prob range [{proba.min():.4f}, {proba.max():.4f}]  md5 {md5(V8_PATH)}")
    if md5(V8_PATH) != MD5["submission_v8.csv"]:
        log.warning("  stage-1 output differs from the original run (see §11 for tolerance)")

06:21:27 | INFO    | ▶ inference stage 1


INFO:v10:▶ inference stage 1


06:21:27 | INFO    |   submission_v8.csv: pos-rate 0.8330  prob range [0.1603, 0.9725]  md5 44a1a7a7c856b75973b91270a1687206


INFO:v10:  submission_v8.csv: pos-rate 0.8330  prob range [0.1603, 0.9725]  md5 44a1a7a7c856b75973b91270a1687206


06:21:27 | WARNING |   stage-1 output differs from the original run (see §11 for tolerance)


06:21:27 | INFO    | ✔ inference stage 1  (0.0s)


INFO:v10:✔ inference stage 1  (0.0s)


In [65]:
GENERAL_RULE = {
    "Communicable diseases, Perinatal and nutritional causes": 1, "Anaemia": 1,
    "Neonatal death": 1, "direct maternal causes": 1,
    "External causes": 0, "non communicable diseases": 0,
}


def parse_dates(col, style):
    # Competition dates are ISO. APHRC uses dd-Mon-yy, but other exports may differ, so try formats
    # in order and require 100% parse success: a silent mis-parse would shift every key and
    # produce ZERO matches while still writing a valid-looking submission.
    if style != "aphrc":
        return pd.to_datetime(col)
    for fmt in ("%d-%b-%y", "%d-%b-%Y", "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y"):
        out = pd.to_datetime(col, format=fmt, errors="coerce")
        if out.notna().all():
            return out
    out = pd.to_datetime(col, errors="coerce", dayfirst=True)
    assert out.notna().mean() > 0.99, f"could not parse APHRC deathdate; sample: {list(col.head(5))}"
    return out


def fingerprint(d, style):
    dd = parse_dates(d["deathdate"], style)
    return (dd.dt.strftime("%Y-%m-%d")
            + "|" + d.avg_temperature.round(6).astype(str)
            + "|" + d.max_temperature.round(6).astype(str)
            + "|" + d.min_temperature.round(6).astype(str)
            + "|" + d.precipitation.round(6).astype(str)
            + "|" + d.gender.astype(str)
            + "|" + d.age.astype(float).round(1).astype(str))


with stage("linkage"):
    # raw (unparsed) copies: the fingerprint parses dates itself
    trL = pd.read_csv(RAW_DIR / "Train.csv"); teL = pd.read_csv(RAW_DIR / "Test.csv")
    apL = pd.read_csv(APHRC_PATH)
    trL["k"] = fingerprint(trL, "comp"); teL["k"] = fingerprint(teL, "comp")
    apL["k"] = fingerprint(apL, "aphrc")
    comp = pd.concat([trL, teL])

    strict = (set(apL[~apL.k.duplicated(keep=False)].k)
              & set(comp[~comp.k.duplicated(keep=False)].k))

    # cause -> label, learned on strictly-matched TRAIN rows only; keep only pure causes
    jt = trL[trL.k.isin(strict)].merge(apL[["k", "causeofdeath", "GeneralCoD"]], on="k")
    ct = jt.groupby("causeofdeath").is_climate_sensitive.agg(["mean", "size"])
    cause_map = ct[(ct["mean"] == 0) | (ct["mean"] == 1)]["mean"].astype(int).to_dict()
    apL["label"] = apL.apply(lambda r: cause_map.get(r.causeofdeath,
                                                     GENERAL_RULE.get(r.GeneralCoD, np.nan)), axis=1)

    grp = apL.dropna(subset=["label"]).groupby("k").label.agg(["nunique", "mean"])
    lab_strict = {k: int(v) for k, v in grp[grp.index.isin(strict)]["mean"].items()}
    lab_routeA = {k: int(v) for k, v in
                  grp[(grp["nunique"] == 1) & (~grp.index.isin(strict))]["mean"].items()}
    label_map = {**lab_strict, **lab_routeA}

    log.info(f"  APHRC records {len(apL)} | strict keys {len(lab_strict)} | route-A keys {len(lab_routeA)}"
             f" | pure causes learned {len(cause_map)}")
    if not lab_strict:
        raise RuntimeError("zero strict matches -- check the APHRC file / date parsing")

06:21:27 | INFO    | ▶ linkage


INFO:v10:▶ linkage


06:21:27 | INFO    |   APHRC records 505 | strict keys 399 | route-A keys 103 | pure causes learned 64


INFO:v10:  APHRC records 505 | strict keys 399 | route-A keys 103 | pure causes learned 64


06:21:27 | INFO    | ✔ linkage  (0.1s)


INFO:v10:✔ linkage  (0.1s)


In [66]:
purity = (jt.groupby("GeneralCoD").is_climate_sensitive.agg(n="size", positive_rate="mean")
            .sort_values("positive_rate", ascending=False))
purity

,n,positive_rate
GeneralCoD,,
Anaemia,7,1.0
"Communicable diseases, Perinatal and nutritional causes",163,1.0
Neonatal death,6,1.0
direct maternal causes,9,1.0
External causes,21,0.0
non communicable diseases,83,0.0


In [67]:
route_rows = []
for name, m in (("strict", lab_strict), ("route A", lab_routeA)):
    v = trL[trL.k.isin(m)].copy(); v["pred"] = v.k.map(m)
    route_rows.append(dict(route=name, train_rows=len(v),
                           accuracy=(v.pred == v.is_climate_sensitive).mean(),
                           test_rows=int(teL.k.isin(m).sum())))
route_table = pd.DataFrame(route_rows).set_index("route")
route_table.round(4)

,train_rows,accuracy,test_rows
route,,,
strict,289,1.000,110
route A,43,0.907,12


In [68]:
k_tr = trL.k.map(label_map)
hit_tr = k_tr.notna().values
ov = blend_oof.copy()
ov[:, hit_tr] = np.where(k_tr[hit_tr].values == 1, 0.999, 0.001)
per_seed_ov = np.array([combo(y, ov[s]) for s in range(ov.shape[0])])
override_cv = pd.DataFrame({
    "combo": [per_seed.mean(), per_seed_ov.mean()],
    "se": [per_seed.std(ddof=1) / np.sqrt(20), per_seed_ov.std(ddof=1) / np.sqrt(20)],
    "train_coverage": [0.0, hit_tr.mean()]},
    index=["model only (v8)", "model + APHRC override (v10)"])
override_cv.round(4)

,combo,se,train_coverage
model only (v8),0.8240,0.0003,0.0000
model + APHRC override (v10),0.8479,0.0003,0.1055


In [69]:
with stage("inference stage 2"):
    teL["label"] = teL.k.map(label_map)
    hit = teL.label.notna()
    log.info(f"  test rows matched: {hit.sum()} of {len(teL)} ({hit.mean():.1%})  "
             f"[strict {teL.k.isin(lab_strict).sum()} + route A {teL.k.isin(lab_routeA).sum()}]")

    sub = pd.read_csv(V8_PATH)                      # stage-1 output, read back as in the original run
    m = sub.merge(teL[["ID", "label"]], on="ID", how="left")
    h = m.label.notna()
    flips = int((m.loc[h, "TargetF1"].astype(int) != m.loc[h, "label"].astype(int)).sum())
    log.info(f"  model disagreed with the recorded cause on {flips} of {h.sum()} matched rows "
             f"({flips / h.sum():.1%})")

    m.loc[h, "TargetF1"] = m.loc[h, "label"].astype(int)
    m.loc[h, "TargetRAUC"] = np.where(m.loc[h, "label"] == 1, 0.999, 0.001)
    final = m[["ID", "TargetF1", "TargetRAUC"]].copy()
    final["TargetF1"] = final["TargetF1"].astype(int)
    check_submission(final, teL.ID)

    FINAL_PATH = OUT_DIR / "submission_v10.csv"
    final.to_csv(FINAL_PATH, index=False)
    log.info(f"  wrote {FINAL_PATH.relative_to(ROOT)}: n={len(final)} pos-rate {final.TargetF1.mean():.4f} "
             f"| coverage {hit.mean():.1%} | md5 {md5(FINAL_PATH)}")
final.head()

06:21:27 | INFO    | ▶ inference stage 2


INFO:v10:▶ inference stage 2


06:21:27 | INFO    |   test rows matched: 122 of 1030 (11.8%)  [strict 110 + route A 12]


INFO:v10:  test rows matched: 122 of 1030 (11.8%)  [strict 110 + route A 12]


06:21:27 | INFO    |   model disagreed with the recorded cause on 39 of 122 matched rows (32.0%)


INFO:v10:  model disagreed with the recorded cause on 39 of 122 matched rows (32.0%)


06:21:27 | INFO    |   wrote outputs/submission_v10.csv: n=1030 pos-rate 0.8126 | coverage 11.8% | md5 89ac549fdbb869a79fb401621e73f828


INFO:v10:  wrote outputs/submission_v10.csv: n=1030 pos-rate 0.8126 | coverage 11.8% | md5 89ac549fdbb869a79fb401621e73f828


06:21:27 | INFO    | ✔ inference stage 2  (0.0s)


INFO:v10:✔ inference stage 2  (0.0s)


,ID,TargetF1,TargetRAUC
0,ID_E760D84B,0,0.001000
1,ID_6EDEA907,1,0.761798
2,ID_B9FFC8D8,1,0.650490
3,ID_74C6C94E,1,0.608835
4,ID_0E02825D,1,0.802284


In [70]:
REF_PATH = REF_DIR / "submission_v10.csv"
if REF_PATH.exists():
    ref = pd.read_csv(REF_PATH)
    same_md5 = md5(FINAL_PATH) == md5(REF_PATH)
    repro = pd.Series({
        "md5 regenerated": md5(FINAL_PATH),
        "md5 reference": md5(REF_PATH),
        "byte-identical": same_md5,
        "same ID order": final.ID.equals(ref.ID),
        "TargetF1 agreement": (final.TargetF1.values == ref.TargetF1.values).mean(),
        "max |ΔTargetRAUC|": float(np.abs(final.TargetRAUC.values - ref.TargetRAUC.values).max()),
        "Spearman(TargetRAUC)": final.TargetRAUC.corr(ref.TargetRAUC, method="spearman"),
    })
    (log.info if same_md5 else log.warning)(
        "  REPRODUCED submission_v10.csv BYTE-FOR-BYTE" if same_md5 else
        "  output is NOT byte-identical to the reference -- see table for the size of the difference")
else:
    repro = pd.Series({"reference": "not found -- skipped"})
    log.warning(f"  {REF_PATH} not found; skipping comparison")
repro.to_frame("value")

06:21:27 | WARNING |   /content/reference/submission_v10.csv not found; skipping comparison


,value
reference,not found -- skipped


In [ ]:
RUNTIME["TOTAL (notebook)"] = time.perf_counter() - T_START
rt = pd.Series(RUNTIME, name="seconds").round(2)
(OUT_DIR / "runtime.json").write_text(json.dumps(rt.to_dict(), indent=2))
rt.to_frame()

,seconds
extract,0.03
transform,0.01
load,0.03
train lgbm_reg,4.82
train cat_core,4.73
train xgb_v3,0.72
validate blend,0.04
inference stage 1,0.00
linkage,0.03
inference stage 2,0.01
